# S6E2 Stacking

Train a meta-learner on out-of-fold (OOF) predictions from CatBoost, LightGBM, XGBoost, TabNet.

**Why this works**: each OOF prediction was made on held-out data, so training a meta-learner on them vs. the true labels has no information leakage. The meta AUC computed on the full training set is a valid, unbiased estimate.

**Strategy**: 
1. Load pre-computed OOF and test arrays
2. Logit-transform (better for linear meta-learners)
3. Try LR, Ridge-logit, and XGB meta-learners
4. Evaluate with 5-fold meta-CV for honest AUC estimates
5. Submit if > 0.95537 (current best ensemble)

## Imports & Data

In [1]:
import numpy as np
import pandas as pd
import subprocess
from pathlib import Path
from scipy.special import logit, expit
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

KAGGLE_DATA = Path('/kaggle/input/playground-series-s6e2')
LOCAL_DATA  = Path('data')
DATA_DIR    = KAGGLE_DATA if KAGGLE_DATA.exists() else LOCAL_DATA
SUB_DIR     = Path('submissions')

train = pd.read_csv(DATA_DIR / 'train.csv')
train.columns = train.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
ss = pd.read_csv(DATA_DIR / 'sample_submission.csv')

y = train['heart_disease'].map({'Absence': 0, 'Presence': 1}).values
print(f'y: {len(y):,} rows, {y.mean():.3f} positive rate')

y: 630,000 rows, 0.448 positive rate


## Load OOF & Test Arrays

In [2]:
oof_cat    = np.load(SUB_DIR / 'oof_cat.npy')
oof_lgb    = np.load(SUB_DIR / 'oof_lgb.npy')
oof_xgb    = np.load(SUB_DIR / 'oof_xgb.npy')
oof_tabnet = np.load(SUB_DIR / 'oof_tabnet.npy')

test_cat    = np.load(SUB_DIR / 'test_cat.npy')
test_lgb    = np.load(SUB_DIR / 'test_lgb.npy')
test_xgb    = np.load(SUB_DIR / 'test_xgb.npy')
test_tabnet = np.load(SUB_DIR / 'test_tabnet.npy')

MODEL_NAMES = ['cat', 'lgb', 'xgb', 'tabnet']
oofs  = [oof_cat,  oof_lgb,  oof_xgb,  oof_tabnet]
tests = [test_cat, test_lgb, test_xgb, test_tabnet]

print('Individual OOF AUCs:')
for name, oof in zip(MODEL_NAMES, oofs):
    print(f'  {name:<8}  {roc_auc_score(y, oof):.5f}')

baseline = roc_auc_score(y, (oof_cat + oof_lgb + oof_xgb) / 3)
print(f'\nBaseline (simple avg 3 GBTs): {baseline:.5f}')

Individual OOF AUCs:
  cat       0.95532


  lgb       0.95523
  xgb       0.95525


  tabnet    0.95369

Baseline (simple avg 3 GBTs): 0.95537


## Build Meta-Feature Matrices

Two versions:
- **raw**: probabilities as-is (good for tree-based meta-learners)
- **logit**: log-odds transform (better calibrated for linear meta-learners)

In [3]:
EPS = 1e-6

def to_logit(arr):
    return logit(np.clip(arr, EPS, 1 - EPS))

# 4-model matrices
M4_raw    = np.column_stack(oofs)
M4_logit  = np.column_stack([to_logit(o) for o in oofs])
T4_raw    = np.column_stack(tests)
T4_logit  = np.column_stack([to_logit(t) for t in tests])

# 3-model (GBTs only)
M3_raw    = np.column_stack(oofs[:3])
M3_logit  = np.column_stack([to_logit(o) for o in oofs[:3]])
T3_raw    = np.column_stack(tests[:3])
T3_logit  = np.column_stack([to_logit(t) for t in tests[:3]])

print(f'Meta-train shape (4-model): {M4_raw.shape}')
print(f'Meta-test shape  (4-model): {T4_raw.shape}')
print(f'\nLogit range for cat: [{to_logit(oof_cat).min():.1f}, {to_logit(oof_cat).max():.1f}]')

Meta-train shape (4-model): (630000, 4)
Meta-test shape  (4-model): (270000, 4)

Logit range for cat: [-8.7, 10.3]


## Meta-Learner CV Evaluation

For each meta-learner:
- 5-fold CV on the OOF features → gives honest meta-AUC estimate with variance
- OOF predictions are already honest (no leakage), so any positive gain is real

In [4]:
meta_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

CANDIDATES = {
    # LR on raw probabilities
    'LR_raw_3gbt':    (LogisticRegression(C=1.0, max_iter=500, random_state=42), M3_raw,   T3_raw),
    'LR_raw_4model':  (LogisticRegression(C=1.0, max_iter=500, random_state=42), M4_raw,   T4_raw),
    # LR on logits — typically sharper, better calibrated
    'LR_logit_3gbt':  (LogisticRegression(C=1.0, max_iter=500, random_state=42), M3_logit, T3_logit),
    'LR_logit_4model':(LogisticRegression(C=1.0, max_iter=500, random_state=42), M4_logit, T4_logit),
    # Stronger regularization — prevents overfitting on near-identical features
    'LR_C01_logit_4': (LogisticRegression(C=0.1, max_iter=500, random_state=42), M4_logit, T4_logit),
    'LR_C10_logit_4': (LogisticRegression(C=10.0, max_iter=500, random_state=42), M4_logit, T4_logit),
    # XGB on raw — can learn non-linear combos
    'XGB_raw_4model': (
        xgb.XGBClassifier(n_estimators=50, max_depth=2, learning_rate=0.1,
                          subsample=0.8, use_label_encoder=False,
                          eval_metric='logloss', device='cuda',
                          random_state=42, verbosity=0),
        M4_raw, T4_raw
    ),
    'XGB_logit_4model': (
        xgb.XGBClassifier(n_estimators=50, max_depth=2, learning_rate=0.1,
                          subsample=0.8, use_label_encoder=False,
                          eval_metric='logloss', device='cuda',
                          random_state=42, verbosity=0),
        M4_logit, T4_logit
    ),
}

results = {}
for name, (model, X_meta, _) in CANDIDATES.items():
    scores = cross_val_score(model, X_meta, y, cv=meta_cv, scoring='roc_auc', n_jobs=-1)
    results[name] = scores
    delta = scores.mean() - baseline
    print(f'{name:<22}  {scores.mean():.5f} ± {scores.std():.5f}  ({delta:+.5f} vs baseline)')

print(f'\nBaseline (simple avg 3 GBTs): {baseline:.5f}')

LR_raw_3gbt             0.95517 ± 0.00063  (-0.00020 vs baseline)


LR_raw_4model           0.95519 ± 0.00063  (-0.00018 vs baseline)


LR_logit_3gbt           0.95538 ± 0.00060  (+0.00001 vs baseline)


LR_logit_4model         0.95540 ± 0.00061  (+0.00003 vs baseline)


LR_C01_logit_4          0.95540 ± 0.00061  (+0.00004 vs baseline)


LR_C10_logit_4          0.95540 ± 0.00061  (+0.00003 vs baseline)


XGB_raw_4model          0.95530 ± 0.00061  (-0.00007 vs baseline)


XGB_logit_4model        0.95530 ± 0.00060  (-0.00007 vs baseline)

Baseline (simple avg 3 GBTs): 0.95537


## Inspect Learned Weights (LR coefficients)

In [5]:
# Train best LR on full OOF data and inspect coefficients
for name, feat_names, X_meta in [
    ('LR_logit_3gbt',  ['cat', 'lgb', 'xgb'],         M3_logit),
    ('LR_logit_4model',['cat', 'lgb', 'xgb', 'tabnet'], M4_logit),
]:
    m = LogisticRegression(C=1.0, max_iter=500, random_state=42)
    m.fit(X_meta, y)
    coefs = m.coef_[0]
    total = coefs.sum()
    print(f'\n{name} — coefficients (normalized):')
    for f, c in zip(feat_names, coefs):
        print(f'  {f:<8}  raw={c:.4f}  share={c/total:.3f}')


LR_logit_3gbt — coefficients (normalized):
  cat       raw=0.6365  share=0.635
  lgb       raw=0.1518  share=0.151
  xgb       raw=0.2149  share=0.214



LR_logit_4model — coefficients (normalized):
  cat       raw=0.7984  share=0.797
  lgb       raw=0.1780  share=0.178
  xgb       raw=0.1716  share=0.171
  tabnet    raw=-0.1464  share=-0.146


## Select Best & Build Test Predictions

In [6]:
# Find best meta-learner by mean CV AUC
best_name = max(results, key=lambda k: results[k].mean())
best_auc  = results[best_name].mean()
best_model, best_X_meta, best_T_meta = CANDIDATES[best_name]

print(f'Best meta-learner: {best_name}')
print(f'  CV AUC = {best_auc:.5f}  ({best_auc - baseline:+.5f} vs baseline)')

if best_auc > baseline:
    best_model.fit(best_X_meta, y)
    test_preds = best_model.predict_proba(best_T_meta)[:, 1]
    print(f'  Test preds: [{test_preds.min():.4f}, {test_preds.max():.4f}]  mean={test_preds.mean():.4f}')
else:
    print('  No improvement over baseline — would use simple avg instead')

Best meta-learner: LR_C01_logit_4
  CV AUC = 0.95540  (+0.00004 vs baseline)


  Test preds: [0.0002, 0.9999]  mean=0.4498


## Also: Full-data OOF AUC (for reference)
Train on all OOF → evaluate on same data. Valid since OOFs are already honest.

In [7]:
print('Full-data OOF AUC (train-on-OOF, eval-on-OOF — valid, no leakage):')
for name, (model, X_meta, _) in CANDIDATES.items():
    model.fit(X_meta, y)
    oof_preds = model.predict_proba(X_meta)[:, 1]
    auc = roc_auc_score(y, oof_preds)
    print(f'  {name:<22}  {auc:.5f}')

Full-data OOF AUC (train-on-OOF, eval-on-OOF — valid, no leakage):


  LR_raw_3gbt             0.95517


  LR_raw_4model           0.95519


  LR_logit_3gbt           0.95538


  LR_logit_4model         0.95541


  LR_C01_logit_4          0.95541


  LR_C10_logit_4          0.95541


  XGB_raw_4model          0.95539


  XGB_logit_4model        0.95539


## Summary & Submission Decision

In [8]:
print('=== CV AUC Summary ===')
for name, scores in sorted(results.items(), key=lambda x: -x[1].mean()):
    delta = scores.mean() - baseline
    marker = ' ← BEST' if name == best_name else ''
    print(f'  {name:<22}  {scores.mean():.5f}  ({delta:+.5f}){marker}')
print(f'  {"baseline_avg3gbt":<22}  {baseline:.5f}  (reference)')

SUBMIT_THRESHOLD = baseline  # only submit if strictly better

if best_auc > SUBMIT_THRESHOLD:
    label   = f'stack_{best_name}'
    fname   = f'submissions/{label}.csv'
    cv_auc  = best_auc
    
    sub = ss.copy()
    sub['Heart Disease'] = test_preds
    sub.to_csv(fname, index=False)
    
    desc = f'{label} | cv_auc={cv_auc:.4f}'
    r = subprocess.run(
        ['kaggle', 'competitions', 'submit', '-c', 'playground-series-s6e2',
         '-f', fname, '-m', desc],
        capture_output=True, text=True
    )
    status = 'submitted' if 'successfully' in r.stdout.lower() else r.stdout.strip()[:100]
    print(f'\n{label}: {status}')
    print(f'  desc: {desc}')
else:
    print(f'\nNo submission — best meta-learner ({best_auc:.5f}) does not beat baseline ({baseline:.5f})')
    print('Recommendation: stick with ensemble_3gbt_slsqp (best known = 0.95356 LB)')

=== CV AUC Summary ===
  LR_C01_logit_4          0.95540  (+0.00004) ← BEST
  LR_logit_4model         0.95540  (+0.00003)
  LR_C10_logit_4          0.95540  (+0.00003)
  LR_logit_3gbt           0.95538  (+0.00001)
  XGB_logit_4model        0.95530  (-0.00007)
  XGB_raw_4model          0.95530  (-0.00007)
  LR_raw_4model           0.95519  (-0.00018)
  LR_raw_3gbt             0.95517  (-0.00020)
  baseline_avg3gbt        0.95537  (reference)



stack_LR_C01_logit_4: submitted
  desc: stack_LR_C01_logit_4 | cv_auc=0.9554
